# 04 – Modeling and Tuning

Evaluate 4 research baselines and 4 machine-learning model families (Logistic Regression, Random Forest, HistGradientBoosting, MLP) under strict chronological validation, and tune hyperparameters via validation PR-AUC.

**Author:** Ummay Maimona Chaman (22301719)  
**Course:** CSE437 Data Science — Individual Project

Produces: `models/forecast_models.joblib`, `models/tuning_results.json`


## 0 · Environment Setup & Colab Dependencies


In [1]:
%pip install -q "pandas>=2.0" "numpy>=1.24" "matplotlib>=3.7" "seaborn>=0.12" "scikit-learn>=1.3" "joblib>=1.3" "jupyter>=1.0" "scipy>=1.10"


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
import os
import json
import time
import math
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Scikit-learn imports
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, ParameterGrid
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay
)
import joblib

warnings.filterwarnings('ignore')
%matplotlib inline

# Aesthetic styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Dynamic repository root detection (seamless in Colab and local Jupyter)
ROOT = Path.cwd().resolve()
for _candidate in [ROOT, *ROOT.parents]:
    if (_candidate / 'data').exists() and (_candidate / 'notebooks').exists():
        ROOT = _candidate
        break

FIGURES   = ROOT / 'figures'
PROCESSED = ROOT / 'data' / 'processed'
MODELS    = ROOT / 'models'
FIGURES.mkdir(exist_ok=True)
PROCESSED.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)

print('Environment configured successfully. All libraries loaded.')
print(f'Working directory: {ROOT.name if ROOT.name else "."}')


Environment configured successfully. All libraries loaded.
Working directory: cse437-japan-earthquake-occurrence-forecasting-ummay-maimona-chaman


## 1 · Chronological Partitioning & Feature Scaling


In [3]:
df_features = pd.read_csv(PROCESSED / 'model_features.csv', parse_dates=['timestamp'])
unique_timestamps = np.sort(df_features['timestamp'].unique())
n_timestamps = len(unique_timestamps)
n_train_times = int(n_timestamps * 0.70)
n_val_times   = int(n_timestamps * 0.15)
train_end_time = unique_timestamps[n_train_times - 1]
val_start_time = unique_timestamps[n_train_times]
val_end_time   = unique_timestamps[n_train_times + n_val_times - 1]
test_start_time = unique_timestamps[n_train_times + n_val_times]

train_mask = df_features['timestamp'] <= train_end_time
val_mask   = (df_features['timestamp'] >= val_start_time) & (df_features['timestamp'] <= val_end_time)
test_mask  = df_features['timestamp'] >= test_start_time

continuous_cols = ['log_event_count_30d', 'log_event_count_lag1', 'max_mag_30d', 'mean_depth_imputed', 'log_cum_energy_30d', 'recent_m45_30d']
grid_cols = [c for c in df_features.columns if c.startswith('grid_') and c != 'grid_id']
df_features[grid_cols] = df_features[grid_cols].astype(float)
feature_set = continuous_cols + grid_cols

X_train_raw = df_features.loc[train_mask, feature_set]
y_train     = df_features.loc[train_mask, 'target_m5_14d'].values.astype(int)
X_val_raw   = df_features.loc[val_mask, feature_set]
y_val       = df_features.loc[val_mask, 'target_m5_14d'].values.astype(int)
X_test_raw  = df_features.loc[test_mask, feature_set]
y_test      = df_features.loc[test_mask, 'target_m5_14d'].values.astype(int)

# Fit StandardScaler strictly on training slice
scaler = StandardScaler()
X_train = X_train_raw.copy()
X_val   = X_val_raw.copy()
X_test  = X_test_raw.copy()
X_train.loc[:, continuous_cols] = scaler.fit_transform(X_train_raw[continuous_cols])
X_val.loc[:, continuous_cols]   = scaler.transform(X_val_raw[continuous_cols])
X_test.loc[:, continuous_cols]  = scaler.transform(X_test_raw[continuous_cols])

print(f'Train partition: {len(X_train)} samples | Target rate: {y_train.mean():.3f}')
print(f'Val partition  : {len(X_val)} samples | Target rate: {y_val.mean():.3f}')
print(f'Test partition : {len(X_test)} samples | Target rate: {y_test.mean():.3f}')


Train partition: 1254 samples | Target rate: 0.249
Val partition  : 264 samples | Target rate: 0.205
Test partition : 276 samples | Target rate: 0.236


## 2 · Multi-Benchmark Baselines & ML Models Suite


In [4]:
# Define Custom Research Baselines
train_baseline_df = df_features.loc[train_mask].copy()
grid_base_rates = train_baseline_df.groupby('grid_id')['target_m5_14d'].mean().to_dict()
grid_poisson_rates = (train_baseline_df.groupby('grid_id')['m5_count_next_14d'].sum() / (train_baseline_df.groupby('grid_id').size() * 14.0)).to_dict()
grid_labels_by_col = {col: col.replace('grid_', '') for col in grid_cols}

class GridProbabilityBaseline:
    def __init__(self, grid_probabilities, grid_columns, probability_from_rate=False):
        self.grid_probabilities = grid_probabilities
        self.grid_columns = grid_columns
        self.probability_from_rate = probability_from_rate
    def fit(self, X, y):
        self.classes_ = np.array([0, 1])
        self.default_probability_ = float(np.mean(y))
        return self
    def predict_proba(self, X):
        active_col = X[self.grid_columns].idxmax(axis=1)
        labels = active_col.map(grid_labels_by_col)
        values = labels.map(self.grid_probabilities).fillna(self.default_probability_).astype(float).to_numpy()
        if self.probability_from_rate:
            values = 1.0 - np.exp(-values * 14.0)
        values = np.clip(values, 0.0, 1.0)
        return np.column_stack([1.0 - values, values])
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

class PersistenceBaseline:
    def fit(self, X, y):
        self.classes_ = np.array([0, 1])
        self.active_risk_ = float(np.mean(y[X['recent_m45_30d'].to_numpy() == 1])) if (X['recent_m45_30d'] == 1).any() else float(np.mean(y))
        self.quiet_risk_ = float(np.mean(y[X['recent_m45_30d'].to_numpy() == 0])) if (X['recent_m45_30d'] == 0).any() else float(np.mean(y))
        return self
    def predict_proba(self, X):
        p = np.where(X['recent_m45_30d'].to_numpy() == 1, self.active_risk_, self.quiet_risk_)
        return np.column_stack([1.0 - p, p])
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

baselines = {
    '1. Dummy Baseline (Global Prior)': DummyClassifier(strategy='prior'),
    '2. Historical Grid Base Rate': GridProbabilityBaseline(grid_base_rates, grid_cols),
    '3. Persistence (Recent M4.5+)': PersistenceBaseline(),
    '4. Poisson Seismicity Rate': GridProbabilityBaseline(grid_poisson_rates, grid_cols, probability_from_rate=True)
}

models = {
    '1. Logistic Regression (L2)': LogisticRegression(penalty='l2', C=1.0, class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
    '2. Random Forest Classifier': RandomForestClassifier(n_estimators=150, max_depth=6, min_samples_leaf=15, class_weight='balanced', random_state=RANDOM_STATE),
    '3. Gradient Boosted Trees': HistGradientBoostingClassifier(max_iter=120, max_depth=4, min_samples_leaf=20, learning_rate=0.03, random_state=RANDOM_STATE),
    '4. Multi-Layer Perceptron (MLP)': MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', alpha=0.01, max_iter=250, random_state=RANDOM_STATE)
}

for name, clf in {**baselines, **models}.items():
    clf.fit(X_train, y_train)
    print(f'Trained: {name}')


Trained: 1. Dummy Baseline (Global Prior)
Trained: 2. Historical Grid Base Rate
Trained: 3. Persistence (Recent M4.5+)
Trained: 4. Poisson Seismicity Rate
Trained: 1. Logistic Regression (L2)


Trained: 2. Random Forest Classifier


Trained: 3. Gradient Boosted Trees


Trained: 4. Multi-Layer Perceptron (MLP)


## 3 · Hyperparameter Tuning via Validation PR-AUC


In [5]:
# Grid Search Optimization for Gradient Boosting on Validation PR-AUC
param_grid = {
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'max_depth': [3, 4, 6],
    'min_samples_leaf': [15, 30, 50],
    'l2_regularization': [0.0, 0.1, 1.0]
}

tuning_records = []
for p in ParameterGrid(param_grid):
    hgb = HistGradientBoostingClassifier(**p, max_iter=150, random_state=RANDOM_STATE).fit(X_train, y_train)
    val_pr = average_precision_score(y_val, hgb.predict_proba(X_val)[:, 1])
    tuning_records.append({**p, 'validation_pr_auc': val_pr})

tuning_df = pd.DataFrame(tuning_records).sort_values('validation_pr_auc', ascending=False)
display(tuning_df.head(10))

best_params = tuning_df.iloc[0].drop('validation_pr_auc').to_dict()
best_params = {k: int(v) if k in ('max_depth', 'min_samples_leaf') else float(v) for k, v in best_params.items()}
print('Best Hyperparameters:', best_params)

# Refit Tuned Gradient Boosting Model
models['3. Gradient Boosted Trees (Tuned)'] = HistGradientBoostingClassifier(**best_params, max_iter=150, random_state=RANDOM_STATE).fit(X_train, y_train)

# Serialize model bundle & tuning history
bundle = {
    'baselines': baselines,
    'models': models,
    'scaler': scaler,
    'continuous_cols': continuous_cols,
    'grid_cols': grid_cols,
    'feature_set': feature_set
}
joblib.dump(bundle, MODELS / 'forecast_models.joblib')
(MODELS / 'tuning_results.json').write_text(json.dumps(tuning_df.to_dict(orient='records'), indent=2))
print('Saved trained model bundle to models/forecast_models.joblib')


,l2_regularization,learning_rate,max_depth,min_samples_leaf,validation_pr_auc
7,0.0,0.01,6,30,0.475383
43,0.1,0.01,6,30,0.472017
8,0.0,0.01,6,50,0.468223
78,1.0,0.01,6,15,0.468105
39,0.1,0.01,4,15,0.466651
6,0.0,0.01,6,15,0.465651
79,1.0,0.01,6,30,0.464876
50,0.1,0.03,4,50,0.464603
42,0.1,0.01,6,15,0.463776
95,1.0,0.05,4,50,0.463697


Best Hyperparameters: {'l2_regularization': 0.0, 'learning_rate': 0.01, 'max_depth': 6, 'min_samples_leaf': 30}


Saved trained model bundle to models/forecast_models.joblib
